<a href="https://colab.research.google.com/github/boba1987/advanced-neural-networks-and-deep-learning/blob/main/Customer_Support_Ticket_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Customer Support Ticket Classification

Predict **ticket type** (`Incident`, `Request`, `Problem`, `Change`) from ticket **`body` text only** using **TF-IDF + PyTorch MLP** and **BERT** (English only).

**Dataset:** `dataset-tickets-en.csv` (~11,923 rows) | **Split:** 80% train / 20% test


## Colab notes

- **Runtime → Run all** (GPU optional for Phases 1–5; **recommended for Phase 6 BERT**).
- Phase 3 runs a **6-config hyperparameter search** (~10–20 min on CPU) before final MLP training.
- Data is fetched with **`curl`** from GitHub.


## Phase 1: Setup and data loading


In [ ]:
!pip install -q torch scikit-learn pandas matplotlib seaborn joblib


In [ ]:
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### Load dataset

Input = **`body` only** (no subject, no queue); label = **type**. English rows only (`language == en`).


In [ ]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/boba1987/advanced-neural-networks-and-deep-learning/main/dataset-tickets-en.csv"
CSV_PATH = "dataset-tickets-en.csv"

!curl -L -o {CSV_PATH} "{GITHUB_RAW_URL}"

raw = pd.read_csv(CSV_PATH)
raw["body"] = raw["body"].astype(str).str.strip()
raw = raw[raw["language"].astype(str).str.lower() == "en"].copy()

df = pd.DataFrame({
    "ticket_text": raw["body"],
    "category": raw["type"],
}).dropna()

df = df[df["ticket_text"].str.len() > 0].reset_index(drop=True)

print(f"Loaded {len(df):,} tickets")
print(f"Categories: {sorted(df['category'].unique())}")
print(df["category"].value_counts())


### Exploratory analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["category"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Tickets per type")
axes[0].tick_params(axis="x", rotation=45)
df["ticket_text"].str.len().hist(bins=50, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Body length")
plt.tight_layout()
plt.show()


## Phase 2: Preprocessing, labels, and split


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text_clean"] = df["ticket_text"].apply(clean_text)

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["category"])
num_classes = len(label_encoder.classes_)
id_to_label = {i: label_encoder.classes_[i] for i in range(num_classes)}
print("Label mapping:", id_to_label)


In [ ]:
texts_raw = df["ticket_text"].values
texts_mlp = df["text_clean"].values
labels = df["label"].values

X_train_full, X_test_orig, y_train_full, y_test, texts_mlp_train_full, texts_mlp_test = train_test_split(
    texts_raw, labels, texts_mlp,
    test_size=0.2, random_state=42, stratify=labels,
)

X_train, X_val, y_train, y_val, X_mlp_train, X_mlp_val = train_test_split(
    X_train_full, y_train_full, texts_mlp_train_full,
    test_size=0.1, random_state=42, stratify=y_train_full,
)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test_orig):,}")


## Phase 3: TF-IDF + MLP

Define the model, then run a small hyperparameter search (validation **macro F1**) before final training in Phase 4.


In [ ]:
EARLY_STOP_PATIENCE = 3
SEARCH_EPOCHS = 12

class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
pin_memory = device.type == "cuda"

class SparseDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse.tocsr()
        self.y = y.astype(np.int64)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        row = self.X[idx].toarray().astype(np.float32).squeeze()
        return torch.from_numpy(row), torch.tensor(self.y[idx], dtype=torch.long)

class TicketClassifier(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes, dropouts):
        super().__init__()
        layers, in_dim = [], input_size
        for h, d in zip(hidden_sizes, dropouts):
            layers.extend([nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(d)])
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

def build_vectorizer(params):
    return TfidfVectorizer(
        max_features=params["max_features"],
        ngram_range=params["ngram_range"],
        min_df=params["min_df"],
        max_df=params["max_df"],
        sublinear_tf=True,
    )

def run_mlp_epoch(model, loader, loss_fn, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        if train:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = loss_fn(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        else:
            with torch.no_grad():
                outputs = model(batch_X)
                loss = loss_fn(outputs, batch_y)

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * batch_y.size(0)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / total
    acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1

def train_mlp_trial(params, max_epochs, verbose=False):
    set_seed(42)
    vectorizer = build_vectorizer(params)
    X_train_sp = vectorizer.fit_transform(X_mlp_train)
    X_val_sp = vectorizer.transform(X_mlp_val)
    input_size = X_train_sp.shape[1]

    train_loader = DataLoader(
        SparseDataset(X_train_sp, y_train),
        batch_size=params["batch_size"], shuffle=True, pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        SparseDataset(X_val_sp, y_val),
        batch_size=params["batch_size"], shuffle=False, pin_memory=pin_memory,
    )

    model = TicketClassifier(
        input_size, num_classes, params["hidden_sizes"], params["dropouts"]
    ).to(device)
    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights_tensor, label_smoothing=params["label_smoothing"]
    )
    optimizer = optim.AdamW(
        model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"]
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    best_val_f1 = 0.0
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        train_loss, train_acc, _ = run_mlp_epoch(
            model, train_loader, loss_fn, optimizer, train=True
        )
        val_loss, val_acc, val_f1 = run_mlp_epoch(
            model, val_loader, loss_fn, train=False
        )
        scheduler.step(val_loss)

        if verbose:
            print(
                f"  epoch {epoch+1:2d}/{max_epochs} | "
                f"val acc {val_acc*100:.2f}% | val F1 {val_f1*100:.2f}%"
            )

        if val_f1 > best_val_f1:
            best_val_f1, best_val_acc = val_f1, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)

    return {
        "name": params["name"],
        "val_f1": best_val_f1,
        "val_acc": best_val_acc,
        "params": params,
        "vectorizer": vectorizer,
        "input_size": input_size,
        "model_state": model.state_dict(),
    }

SEARCH_CONFIGS = [
    {
        "name": "baseline",
        "max_features": 5000, "ngram_range": (1, 3), "min_df": 2, "max_df": 0.95,
        "hidden_sizes": (256, 128, 64), "dropouts": (0.5, 0.4, 0.3),
        "lr": 1e-3, "weight_decay": 1e-3, "label_smoothing": 0.1,
        "batch_size": 64, "max_epochs": 15,
    },
    {
        "name": "more_features",
        "max_features": 10000, "ngram_range": (1, 3), "min_df": 2, "max_df": 0.95,
        "hidden_sizes": (256, 128, 64), "dropouts": (0.5, 0.4, 0.3),
        "lr": 1e-3, "weight_decay": 1e-3, "label_smoothing": 0.1,
        "batch_size": 64, "max_epochs": 25,
    },
    {
        "name": "wider_net_lower_lr",
        "max_features": 10000, "ngram_range": (1, 3), "min_df": 2, "max_df": 0.95,
        "hidden_sizes": (512, 256, 128), "dropouts": (0.3, 0.2, 0.1),
        "lr": 5e-4, "weight_decay": 1e-3, "label_smoothing": 0.1,
        "batch_size": 64, "max_epochs": 25,
    },
    {
        "name": "wider_net_small_batch",
        "max_features": 10000, "ngram_range": (1, 3), "min_df": 2, "max_df": 0.95,
        "hidden_sizes": (512, 256, 128), "dropouts": (0.3, 0.2, 0.1),
        "lr": 5e-4, "weight_decay": 1e-4, "label_smoothing": 0.05,
        "batch_size": 32, "max_epochs": 25,
    },
    {
        "name": "longer_ngrams",
        "max_features": 8000, "ngram_range": (1, 4), "min_df": 2, "max_df": 0.90,
        "hidden_sizes": (512, 256, 128), "dropouts": (0.3, 0.2, 0.1),
        "lr": 5e-4, "weight_decay": 1e-3, "label_smoothing": 0.1,
        "batch_size": 64, "max_epochs": 25,
    },
    {
        "name": "max_features_15k",
        "max_features": 15000, "ngram_range": (1, 3), "min_df": 2, "max_df": 0.95,
        "hidden_sizes": (512, 256, 128), "dropouts": (0.3, 0.2, 0.1),
        "lr": 5e-4, "weight_decay": 1e-3, "label_smoothing": 0.1,
        "batch_size": 64, "max_epochs": 25,
    },
]

print(f"Defined MLP helpers and {len(SEARCH_CONFIGS)} search configs.")


### Hyperparameter search

Each config is trained for up to `SEARCH_EPOCHS` with early stopping. The winner is chosen by **validation macro F1**.

In [ ]:
search_results = []
print(f"Running {len(SEARCH_CONFIGS)} configs (up to {SEARCH_EPOCHS} epochs each)...\n")

for i, cfg in enumerate(SEARCH_CONFIGS, 1):
    print(f"[{i}/{len(SEARCH_CONFIGS)}] {cfg['name']}")
    result = train_mlp_trial(cfg, max_epochs=SEARCH_EPOCHS, verbose=False)
    search_results.append(result)
    print(
        f"  val acc {result['val_acc']*100:.2f}% | "
        f"val macro F1 {result['val_f1']*100:.2f}% | "
        f"features {result['input_size']:,}\n"
    )

search_results.sort(key=lambda r: r["val_f1"], reverse=True)
best_trial = search_results[0]
BEST_PARAMS = best_trial["params"]

print("Search results (sorted by val macro F1):")
print(f"{'Config':<22} {'Val Acc':>10} {'Val F1':>10} {'Features':>10}")
print("-" * 54)
for r in search_results:
    print(
        f"{r['name']:<22} {r['val_acc']*100:>9.2f}% "
        f"{r['val_f1']*100:>9.2f}% {r['input_size']:>10,}"
    )

print(f"\nBest config: {BEST_PARAMS['name']}")
print(
    f"max_features={BEST_PARAMS['max_features']}, "
    f"ngrams={BEST_PARAMS['ngram_range']}, "
    f"hidden={BEST_PARAMS['hidden_sizes']}, "
    f"lr={BEST_PARAMS['lr']}, batch={BEST_PARAMS['batch_size']}"
)

In [ ]:
HIDDEN_SIZES = BEST_PARAMS["hidden_sizes"]
DROPOUTS = BEST_PARAMS["dropouts"]
MAX_FEATURES = BEST_PARAMS["max_features"]
BATCH_SIZE = BEST_PARAMS["batch_size"]
MAX_EPOCHS = BEST_PARAMS["max_epochs"]
MLP_LR = BEST_PARAMS["lr"]
MLP_WEIGHT_DECAY = BEST_PARAMS["weight_decay"]
LABEL_SMOOTHING = BEST_PARAMS["label_smoothing"]

text_vectorizer = build_vectorizer(BEST_PARAMS)
X_train_sparse = text_vectorizer.fit_transform(X_mlp_train)
X_val_sparse = text_vectorizer.transform(X_mlp_val)
X_test_sparse = text_vectorizer.transform(texts_mlp_test)
input_size = X_train_sparse.shape[1]

train_loader = DataLoader(
    SparseDataset(X_train_sparse, y_train),
    batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory,
)
val_loader = DataLoader(
    SparseDataset(X_val_sparse, y_val),
    batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory,
)
test_loader = DataLoader(
    SparseDataset(X_test_sparse, y_test),
    batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory,
)

model = TicketClassifier(input_size, num_classes, HIDDEN_SIZES, DROPOUTS).to(device)
loss_fn = nn.CrossEntropyLoss(
    weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING
)
optimizer = optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

history = {
    "train_loss": [], "train_acc": [], "train_f1": [],
    "val_loss": [], "val_acc": [], "val_f1": [],
}
best_val_f1 = 0.0
best_val_acc = 0.0
best_epoch = 0
epochs_no_improve = 0
best_state = None

print(f"Using tuned config: {BEST_PARAMS['name']}")
print(f"TF-IDF features: {input_size:,}")
print(f"MLP parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training for up to {MAX_EPOCHS} epochs in Phase 4...")

## Phase 4: Training and evaluation


In [ ]:
print(f"Training TF-IDF + MLP on {device} (config: {BEST_PARAMS['name']})...\n")
for epoch in range(MAX_EPOCHS):
    train_loss, train_acc, train_f1 = run_mlp_epoch(
        model, train_loader, loss_fn, optimizer, train=True
    )
    val_loss, val_acc, val_f1 = run_mlp_epoch(
        model, val_loader, loss_fn, train=False
    )
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch [{epoch+1:2d}/{MAX_EPOCHS}] | "
        f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% F1 {train_f1*100:.2f}% | "
        f"val loss {val_loss:.4f} acc {val_acc*100:.2f}% F1 {val_f1*100:.2f}% | "
        f"lr {optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1, best_val_acc, best_epoch, epochs_no_improve = val_f1, val_acc, epoch + 1, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

print(f"\nBest epoch {best_epoch}, val acc {best_val_acc*100:.2f}%, val macro F1 {best_val_f1*100:.2f}%")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ep = range(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="Train")
axes[0].plot(ep, history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[1].plot(ep, [a * 100 for a in history["train_acc"]], label="Train")
axes[1].plot(ep, [a * 100 for a in history["val_acc"]], label="Val")
axes[1].set_title("Accuracy (%)")
axes[2].plot(ep, [a * 100 for a in history["train_f1"]], label="Train")
axes[2].plot(ep, [a * 100 for a in history["val_f1"]], label="Val")
axes[2].set_title("Macro F1 (%)")
for ax in axes:
    ax.legend()
plt.suptitle(f"Training curves — TF-IDF + MLP ({BEST_PARAMS['name']})")
plt.tight_layout()
plt.show()


In [ ]:
def predict_loader(loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch_X, labels in loader:
            logits = model(batch_X.to(device))
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.append(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.vstack(all_probs)

y_pred, y_true, y_probs = predict_loader(test_loader)
test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro")
top_k = min(3, num_classes)
topk_acc = np.mean([y_true[i] in np.argsort(y_probs[i])[-top_k:] for i in range(len(y_true))])

print(f"Test accuracy: {test_acc*100:.2f}%")
print(f"Test macro F1: {test_f1*100:.2f}%")
print(f"Top-{top_k} accuracy: {topk_acc*100:.2f}%")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion matrix — TF-IDF + MLP")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()


## Phase 5: Save and inference


In [ ]:
torch.save({
    "state_dict": model.state_dict(),
    "input_size": input_size,
    "num_classes": num_classes,
    "hidden_sizes": HIDDEN_SIZES,
    "dropouts": DROPOUTS,
    "max_features": MAX_FEATURES,
    "ngram_range": BEST_PARAMS["ngram_range"],
    "config_name": BEST_PARAMS["name"],
    "model_type": "tfidf_mlp",
}, "ticket_classifier.pth")
joblib.dump(text_vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
print(f"Saved tuned model ({BEST_PARAMS['name']}): ticket_classifier.pth, tfidf_vectorizer.pkl, label_encoder.pkl")


In [ ]:
def predict_ticket(description, top_k=None):
    """Predict type from ticket body text."""
    if top_k is None:
        top_k = num_classes
    model.eval()
    clean = clean_text(description)
    vec = text_vectorizer.transform([clean])
    t = torch.tensor(vec.toarray(), dtype=torch.float32).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(t), dim=1).cpu().numpy()[0]
    top_idx = np.argsort(probs)[-top_k:][::-1]
    snippet = description[:200] + ("..." if len(description) > 200 else "")
    print(snippet)
    print("-" * 50)
    for rank, idx in enumerate(top_idx, 1):
        print(f"{rank}. {id_to_label[idx]:<35} {probs[idx]*100:.2f}%")
    print()


In [ ]:
predict_ticket("The analytics platform crashed and is down — we need it restored immediately.")
predict_ticket("Can you provide documentation on how to configure two-factor authentication?")
predict_ticket("Recurring sync errors between systems after the last software update.")
predict_ticket("We plan to migrate the database next month and need approval for the change window.")

rng = np.random.default_rng(42)
for i in rng.choice(len(X_test_orig), size=2, replace=False):
    print("--- Random test ticket ---")
    predict_ticket(X_test_orig[i])


## Phase 6: BERT fine-tuning (comparison)

Fine-tune **`bert-base-uncased`** on the **same train/val/test split** as the MLP baseline, then compare test metrics side by side.

In [ ]:
!pip install -q transformers

from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN = 256
BERT_BATCH_SIZE = 16 if device.type == "cuda" else 8
BERT_MAX_EPOCHS = 3
BERT_LR = 2e-5
BERT_WARMUP_RATIO = 0.1
BERT_EARLY_STOP_PATIENCE = 2

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=num_classes
).to(device)

class TicketTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels.astype(np.int64)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

bert_train_ds = TicketTextDataset(X_mlp_train, y_train, bert_tokenizer, BERT_MAX_LEN)
bert_val_ds = TicketTextDataset(X_mlp_val, y_val, bert_tokenizer, BERT_MAX_LEN)
bert_test_ds = TicketTextDataset(texts_mlp_test, y_test, bert_tokenizer, BERT_MAX_LEN)

bert_train_loader = DataLoader(bert_train_ds, batch_size=BERT_BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
bert_val_loader = DataLoader(bert_val_ds, batch_size=BERT_BATCH_SIZE, shuffle=False, pin_memory=pin_memory)
bert_test_loader = DataLoader(bert_test_ds, batch_size=BERT_BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

bert_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
bert_optimizer = optim.AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = len(bert_train_loader) * BERT_MAX_EPOCHS
warmup_steps = int(total_steps * BERT_WARMUP_RATIO)
bert_scheduler = get_linear_schedule_with_warmup(
    bert_optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

bert_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
bert_best_val_acc = 0.0
bert_best_epoch = 0
bert_epochs_no_improve = 0
bert_best_state = None

print(f"BERT model: {BERT_MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")
print(f"Batch size: {BERT_BATCH_SIZE} | Max length: {BERT_MAX_LEN}")

In [ ]:
def run_bert_epoch(loader, train=True):
    bert_model.train() if train else bert_model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        if train:
            bert_optimizer.zero_grad()
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            loss = bert_loss_fn(outputs.logits, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)
            bert_optimizer.step()
            bert_scheduler.step()
        else:
            with torch.no_grad():
                outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
                loss = bert_loss_fn(outputs.logits, labels_batch)

        preds = outputs.logits.argmax(dim=1)
        total_loss += loss.item() * labels_batch.size(0)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

    return total_loss / total, correct / total

print(f"Training BERT on {device}...\n")
for epoch in range(BERT_MAX_EPOCHS):
    train_loss, train_acc = run_bert_epoch(bert_train_loader, train=True)
    val_loss, val_acc = run_bert_epoch(bert_val_loader, train=False)

    bert_history["train_loss"].append(train_loss)
    bert_history["train_acc"].append(train_acc)
    bert_history["val_loss"].append(val_loss)
    bert_history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch+1:2d}/{BERT_MAX_EPOCHS}] | "
        f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% | "
        f"val loss {val_loss:.4f} acc {val_acc*100:.2f}% | "
        f"lr {bert_optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_acc > bert_best_val_acc:
        bert_best_val_acc = val_acc
        bert_best_epoch = epoch + 1
        bert_epochs_no_improve = 0
        bert_best_state = {k: v.cpu().clone() for k, v in bert_model.state_dict().items()}
    else:
        bert_epochs_no_improve += 1
        if bert_epochs_no_improve >= BERT_EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

if bert_best_state is not None:
    bert_model.load_state_dict(bert_best_state)
    bert_model.to(device)

print(f"\nBest epoch {bert_best_epoch}, val acc {bert_best_val_acc*100:.2f}%")

In [ ]:
def predict_bert_loader(loader):
    bert_model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["labels"]
            logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.numpy())
            all_probs.append(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.vstack(all_probs)

bert_y_pred, bert_y_true, bert_y_probs = predict_bert_loader(bert_test_loader)
bert_test_acc = accuracy_score(bert_y_true, bert_y_pred)
bert_test_f1 = f1_score(bert_y_true, bert_y_pred, average="macro")
bert_topk_acc = np.mean([
    bert_y_true[i] in np.argsort(bert_y_probs[i])[-top_k:]
    for i in range(len(bert_y_true))
])

print(f"BERT test accuracy: {bert_test_acc*100:.2f}%")
print(f"BERT test macro F1: {bert_test_f1*100:.2f}%")
print(f"BERT top-{top_k} accuracy: {bert_topk_acc*100:.2f}%")
print("\nClassification report:")
print(classification_report(bert_y_true, bert_y_pred, target_names=label_encoder.classes_))

print("\nModel comparison (test set)")
print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10} {f'Top-{top_k}':>10}")
print("-" * 52)
print(f"{'TF-IDF + MLP':<20} {test_acc*100:>9.2f}% {test_f1*100:>9.2f}% {topk_acc*100:>9.2f}%")
print(f"{'BERT':<20} {bert_test_acc*100:>9.2f}% {bert_test_f1*100:>9.2f}% {bert_topk_acc*100:>9.2f}%")
delta_acc = (bert_test_acc - test_acc) * 100
delta_f1 = (bert_test_f1 - test_f1) * 100
print(f"\nBERT vs MLP: {delta_acc:+.2f} pp accuracy, {delta_f1:+.2f} pp macro F1")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ep = range(1, len(bert_history["train_loss"]) + 1)
axes[0].plot(ep, bert_history["train_loss"], label="Train")
axes[0].plot(ep, bert_history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[1].plot(ep, [a * 100 for a in bert_history["train_acc"]], label="Train")
axes[1].plot(ep, [a * 100 for a in bert_history["val_acc"]], label="Val")
axes[1].set_title("Accuracy (%)")
for ax in axes:
    ax.legend()
plt.suptitle("Training curves — BERT")
plt.tight_layout()
plt.show()

bert_cm = confusion_matrix(bert_y_true, bert_y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    bert_cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
)
plt.title("Confusion matrix — BERT")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
def predict_ticket_bert(description, top_k=None):
    """Predict type from ticket body text using fine-tuned BERT."""
    if top_k is None:
        top_k = num_classes
    bert_model.eval()
    clean = clean_text(description)
    encoding = bert_tokenizer(
        clean,
        truncation=True,
        padding="max_length",
        max_length=BERT_MAX_LEN,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    with torch.no_grad():
        probs = torch.softmax(bert_model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=1)
        probs = probs.cpu().numpy()[0]
    top_idx = np.argsort(probs)[-top_k:][::-1]
    snippet = description[:200] + ("..." if len(description) > 200 else "")
    print(snippet)
    print("-" * 50)
    for rank, idx in enumerate(top_idx, 1):
        print(f"{rank}. {id_to_label[idx]:<35} {probs[idx]*100:.2f}%")
    print()

print("--- BERT predictions on sample tickets ---")
predict_ticket_bert("The analytics platform crashed and is down — we need it restored immediately.")
predict_ticket_bert("Can you provide documentation on how to configure two-factor authentication?")
predict_ticket_bert("Recurring sync errors between systems after the last software update.")
predict_ticket_bert("We plan to migrate the database next month and need approval for the change window.")

In [ ]:
BERT_SAVE_DIR = "bert_ticket_classifier"
bert_model.save_pretrained(BERT_SAVE_DIR)
bert_tokenizer.save_pretrained(BERT_SAVE_DIR)
joblib.dump(label_encoder, f"{BERT_SAVE_DIR}/label_encoder.pkl")
print(f"Saved BERT model, tokenizer, and label encoder to {BERT_SAVE_DIR}/")